In [ ]:
%sql
-- Use your Unity Catalog
USE CATALOG ishi_catalog;

-- Create Gold schema if not exists
CREATE SCHEMA IF NOT EXISTS gold;


In [ ]:
from pyspark.sql.functions import col, lit, coalesce, upper, trim, weekofyear, month, year
from pyspark.sql import functions as F


In [ ]:
# Silver tables
pos = spark.table("ishi_catalog.silver.pos")
stores = spark.table("ishi_catalog.silver.stores")
products = spark.table("ishi_catalog.silver.products")
holidays = spark.table("ishi_catalog.silver.holidays")
weather = spark.table("ishi_catalog.silver.weather")
inventory = spark.table("ishi_catalog.silver.inventory")


In [ ]:
pos = pos.withColumn("date", F.to_date(col("date"), "yyyy-MM-dd"))
weather = weather.withColumn("date", F.to_date(col("date"), "yyyy-MM-dd"))
holidays = holidays.withColumn("date", F.to_date(col("date"), "yyyy-MM-dd"))
stores = stores.withColumn("opening_date", F.to_date(col("opening_date"), "yyyy-MM-dd"))


In [ ]:
# 6. Add Revenue Column
pos = pos.withColumn("revenue", col("price") * col("units_sold"))

## Create Sales Fact Table

In [ ]:
sales_fact = (
    pos
    .join(stores, "store_id", "left")
    .join(products, "sku_id", "left")
    .join(holidays, "date", "left")
    .join(weather, ["region", "date"], "left")
    .withColumn("event_name", coalesce(col("event_name"), lit("NO_HOLIDAY")))
    .withColumn("temperature_c", coalesce(col("temperature_c"), lit(0.0)))
    .withColumn("rainfall_mm", coalesce(col("rainfall_mm"), lit(0.0)))
)

(sales_fact.write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("ishi_catalog.gold.sales_fact"))

print("Gold Sales Fact table created!")

Gold Sales Fact table created!


## Create Inventory Fact Table

In [ ]:
inventory_fact = (
    inventory
    .join(stores, "store_id", "left")
    .join(products, "sku_id", "left")
)

(inventory_fact.write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("ishi_catalog.gold.inventory_fact"))

print("Gold Inventory Fact table created!")

Gold Inventory Fact table created!


##  Create Dimension Tables

In [ ]:
products.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ishi_catalog.gold.product_dim")

stores.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ishi_catalog.gold.store_dim")

print("Gold Dimension tables created!")

Gold Dimension tables created!


## Aggregated Sales Tables

In [ ]:
daily_sales = (
    sales_fact.groupBy("date", "store_id", "sku_id", "region", "category")
    .agg(
        F.sum("units_sold").alias("total_units_sold"),
        F.sum("revenue").alias("total_revenue"),
        F.sum("promo_flag").alias("promo_count")
    )
)
daily_sales.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ishi_catalog.gold.daily_sales")

# Weekly
weekly_sales = (
    sales_fact
    .withColumn("week", weekofyear(col("date")))
    .withColumn("year", year(col("date")))
    .groupBy("year", "week", "store_id", "sku_id", "region", "category")
    .agg(
        F.sum("units_sold").alias("total_units_sold"),
        F.sum("revenue").alias("total_revenue"),
        F.sum("promo_flag").alias("promo_count")
    )
)
weekly_sales.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ishi_catalog.gold.weekly_sales")

# Monthly
monthly_sales = (
    sales_fact
    .withColumn("month", month(col("date")))
    .withColumn("year", year(col("date")))
    .groupBy("year", "month", "store_id", "sku_id", "region", "category")
    .agg(
        F.sum("units_sold").alias("total_units_sold"),
        F.sum("revenue").alias("total_revenue"),
        F.sum("promo_flag").alias("promo_count")
    )
)
monthly_sales.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ishi_catalog.gold.monthly_sales")

print("Aggregated Daily / Weekly / Monthly Sales created successfully!")

Aggregated Daily / Weekly / Monthly Sales created successfully!
